  <style>
    .container {
      display: flex;
      gap: 10px; /* Optional spacing between divs */
    }

    .box h1 {
      font-size: 50px;
      padding-top: 10px;
      font-weight: 600;
    }
  </style>
  <div class="container" style="">
    <div class="box">
      <h1>Planning for Robotics</h1>
    </div>
    <div class="box" align="right">
      <img src="images/template/course_logo.svg" alt="Planning for Robotics" height="70"/>
      &nbsp;&nbsp;&nbsp;&nbsp;
      <img src="images/template/logo_i6.svg" alt="Chair Logo" height="70"/>
      &nbsp;&nbsp;&nbsp;&nbsp;
    </div>
  </div>

# Exercise Sheet 5 - Planning and Execution

## Introduction
The time has come, we finally apply many of the concepts and tools we learned in a simulation environment and thus put them to practice. We've already learned about PyRoboSim at the beginning of this course. Now you will build on top of it, to solve some problems using task planning and motion planning. In this exercise we will subsequently develop more complex behaviour for the robot in our simulation and use task planning for the decision making.

The goal of this sheet is to bring together environment modeling, low-level execution, symbolic reasoning, and high-level planning in a coherent loop — all within a simplified but realistic office-world scenario. Toward the end of the sheet, you will even simulate dynamic problem generation: using the robot to discover information in the world and then re-planning based on it.

We also won't be working in Jupyter Notebooks anymore for this task. Create your own project files and upload them as a zip when you're done.

### Learning Objectives
- Basic usage of PyRoboSim
- Modelling an environment for PyRoboSim
- Setting up a planning domain to solve a true robotics problem
- Combining plan generation with execution
- (Basic usage of ROS2, stretch goal)


### Acknowledgments
Huge shoutout to Sebastian Castro for the development of PyRoboSim.

## Task 1: Setting up a PyRoboSim Environment
Before a robot can act, it must have some world to operate in. In this first task, you will define such a world — one that mimics a simplified version of our research group’s office layout.

Using PyRoboSim’s YAML-based world description format, define a small environment that includes multiple rooms (such as `office1`, `hallway`, `kitchen`), along with named locations within those rooms (e.g., desks, coffee machine, robot charging station). Your environment should also include logical connections between rooms to allow the robot to navigate through doors or hallways.

Once defined, load your world into PyRoboSim and verify that it looks and behaves as expected using the built-in visualization. This will serve as the simulation stage for all remaining tasks in this sheet.

You can use the very well written documentation of PyRoboSim as your reference https://pyrobosim.readthedocs.io/en/latest/index.html.

### Please see task1 file and run demo.py

## Task 2: Robot Control via the Python API

Now that your robot has a world, it’s time to make it move. PyRoboSim exposes a Python API that allows you to control the robot at a high level (e.g., pick up items, moving around, sensing).

In this task, you will write a short Python script that:
- Loads your custom environment.
- Moves the robot from its starting location to a specified target location (e.g., a desk).
- Picks up a predefined object.
- Carries the object to another room (e.g., the kitchen) and places it on a designated surface.

This task is very similar to what you would commonly find in mobile manipulation routines and provides you with a first interface to the robot’s capabilities. You’ll also start thinking about how to represent actions and goals symbolically.

### Please see the task2 file and run demo.py

## Task 3: Defining a Symbolic Planning Domain

Planning in robotics requires abstracting from the continuous world to a symbolic representation that planners can reason about. In this task, you will design such a symbolic domain using the PDDL modelling skills you developed in the previous exercise. 

Based on the actions available in PyRoboSim, define a symbolic domain that includes operators such as:
- Moving between locations (move).
- Picking up and placing objects (pick, place).

You should then write a simple planning problem in this domain: for instance, picking up a known object and bringing it to a goal location.
Once your domain and problem are written, you can use the Unified Planning Framework (UPF) to load them and verify that a plan can be found. Consider that your actions closely match the behaviour of the actual capabilities of the robot.

In [1]:
from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner

reader = PDDLReader()
problem = reader.parse_problem("task3/domain_task3.pddl", "task3/problem_task3.pddl")

# Let UP auto-pick any installed oneshot planner:
with OneshotPlanner(problem_kind=problem.kind) as planner:
    print("Chosen engine:", planner.name)
    result = planner.solve(problem)
    print("Status:", result.status)
    if result.plan:
        for a in result.plan.actions:
            print(a)


NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 8 of `/tmp/ipykernel_50785/2866678179.py`, you are using the following planning engine:
  * Engine name: pyperplan
  * Developers:  Albert-Ludwigs-Universität Freiburg (Yusra Alkhazraji, Matthias Frorath, Markus Grützner, Malte Helmert, Thomas Liebetraut, Robert Mattmüller, Manuela Ortlieb, Jendrik Seipp, Tobias Springenberg, Philip Stahl, Jan Wülfing)
  * Description: Pyperplan is a lightweight STRIPS planner written in Python.

Chosen engine: Pyperplan
Status: PlanGenerationResultStatus.SOLVED_SATISFICING
move(my_robot, kitchen, hall)
move(my_robot, hall, bedroom)
pick(my_robot, apple1, table1, bedroom)
move(my_robot, bedroom, hall)
put(my_robot, apple1, counter1, hall)


In [2]:
from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner

reader = PDDLReader()
problem = reader.parse_problem("task3/domain_task3.pddl", "task3/problem_task3.pddl")

with OneshotPlanner(name="pyperplan", problem_kind=problem.kind) as planner:
    result = planner.solve(problem)
    assert result.plan is not None

plan_actions = []
for ai in result.plan.actions:
    # ai is an ActionInstance: name + tuple of objects
    plan_actions.append((ai.action.name, [str(o) for o in ai.actual_parameters]))
plan_actions


  *** Credits ***
  * In operation mode `OneshotPlanner` at line 7 of `/tmp/ipykernel_50785/1959372924.py`, you are using the following planning engine:
  * Engine name: pyperplan
  * Developers:  Albert-Ludwigs-Universität Freiburg (Yusra Alkhazraji, Matthias Frorath, Markus Grützner, Malte Helmert, Thomas Liebetraut, Robert Mattmüller, Manuela Ortlieb, Jendrik Seipp, Tobias Springenberg, Philip Stahl, Jan Wülfing)
  * Description: Pyperplan is a lightweight STRIPS planner written in Python.



[('move', ['my_robot', 'kitchen', 'hall']),
 ('move', ['my_robot', 'hall', 'bedroom']),
 ('pick', ['my_robot', 'apple1', 'table1', 'bedroom']),
 ('move', ['my_robot', 'bedroom', 'hall']),
 ('put', ['my_robot', 'apple1', 'counter1', 'hall'])]

## Task 4: Planning and Executing a Visit-All Task

Now that you have symbolic planning in place, let’s put it to use. In many robotic applications, the robot is required to systematically cover or inspect all areas, be it for cleaning, security, or environment discovery.

In this task, you will formulate a symbolic planning problem in which the robot must visit all rooms in your environment. You may use predicates such as `visited(room)` or encode the coverage goal implicitly depending on how you define your planning problem. Use the UPF again to compute a valid plan for this task. Then, implement a simple plan executor in Python that parses the generated plan and issues the corresponding commands (e.g., navigation) to the robot via the PyRoboSim API. This task demonstrates how to close the loop from abstract planning back to concrete execution. You might notice that any error in your modelling will lead to problems during the execution (and vice versa!).

In [3]:
from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner

# Optional: silence the credits banner
import unified_planning as up
up.shortcuts.get_environment().credits_stream = None

reader = PDDLReader()
problem = reader.parse_problem("task4/visit_all_domain.pddl", "task4/visit_all_problem.pddl")

with OneshotPlanner(name="pyperplan", problem_kind=problem.kind) as planner:
    result = planner.solve(problem)
    print("Status:", result.status)
    if result.plan is None:
        raise RuntimeError("No plan found.")
    for action in result.plan.actions:
        print("action made: ", action)

# Extract a clean list of steps you can execute
plan_steps = []
for ai in result.plan.actions:  # ActionInstance
    name = ai.action.name
    params = [str(o) for o in ai.actual_parameters]
    plan_steps.append((name, params))

plan_steps


Status: PlanGenerationResultStatus.SOLVED_SATISFICING
action made:  move(my_robot, kitchen, living_room)
action made:  move(my_robot, living_room, bedroom)
action made:  move(my_robot, bedroom, hall)


[('move', ['my_robot', 'kitchen', 'living_room']),
 ('move', ['my_robot', 'living_room', 'bedroom']),
 ('move', ['my_robot', 'bedroom', 'hall'])]

### To see the difference when the plan is found and not found.

In [4]:
from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner

# Optional: silence the credits banner
import unified_planning as up
up.shortcuts.get_environment().credits_stream = None

reader = PDDLReader()
problem = reader.parse_problem("task4/visit_all_domain.pddl", "task4/visit_all_problem2.pddl")

with OneshotPlanner(name="pyperplan", problem_kind=problem.kind) as planner:
    result = planner.solve(problem)
    print("Status:", result.status)
    if result.plan is None:
        raise RuntimeError("No plan found.")
    for action in result.plan.actions:
        print("action made: ", action)

# Extract a clean list of steps you can execute
plan_steps = []
for ai in result.plan.actions:  # ActionInstance
    name = ai.action.name
    params = [str(o) for o in ai.actual_parameters]
    plan_steps.append((name, params))

plan_steps


Status: PlanGenerationResultStatus.UNSOLVABLE_INCOMPLETELY


RuntimeError: No plan found.

### Please see the task4 file and run task4_visit_all.py

### NOTE: Please also see task4 file for implementation with PyRoboSim

## Task 5: Exploration and Dynamic Planning Problem Generation

Robots often operate in environments where not all information is known beforehand. This final core task of this exercise sheet introduces dynamic problem generation based on runtime observations. With such a tool, you can adapt your behaviour, based on the observations you make. 

You will start by populating your environment with several objects, placed randomly across different rooms. Then:
1. Plan and execute a strategy for the robot to visit each room and record which objects it finds.
2. Use this information to automatically generate a new planning problem: *the robot should now pick up all discovered objects and deliver them to a predefined drop location (e.g., a desk or box in the hallway).*
3. Solve this new problem using UPF and execute the resulting plan.

By the end of this task, you’ll have built a simple but complete autonomous loop: environment exploration → knowledge acquisition → planning → execution.


### Please see the task5 file.

## Optional Task 6: ROS 2 Integration

This optional task invites you to replicate parts of this exercise sheet using PyRoboSim’s ROS 2 interface.

Using a guide you'll be able to find on moodle soon, you should try to reimplement the behaviour of task 5 using the ROS2 interface of PyRoboSim instead of the Python interface:
- Launch PyRoboSim as a ROS 2 node.
- Control the robot using ROS 2 messages and services.
- Integrate symbolic planning with ROS 2-based execution.
- Your planning integration can probably stay very similar to what you currently have.

While this task is not required, it will provide you with a good excuse to get into ROS2. 

### Sorry for not implementing the ROS2 part. We didn't have enough time... :(

## Submission Checklist
- [ ] All of the coding tasks are completed
- [ ] You can show the visualisations and animations as requested
- [ ] You can confidently provide answers to the questions posed in the tasks